<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA3/IL3.1/3_1_4_Laboratorio_3_Evaluacion_Optimizacion_MercadoAndes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ISY0101 · Ingeniería de Soluciones con IA
## EA3 · Actividad 1 — Laboratorio 3: Evaluación y optimización del agente MercadoAndes

**Contexto:** empresa ficticia MercadoAndes. Los datos y los escenarios de negocio son didácticos.

En el Laboratorio 1 registramos eventos y en el Laboratorio 2 examinamos trazas y *spans*. Ahora utilizaremos esa evidencia para **comparar dos implementaciones** y decidir si una modificación mejora el rendimiento del sistema.

**Al finalizar podrás:** definir KPIs, ejecutar pruebas comparables, detectar operaciones redundantes, analizar latencia y uso de recursos, y emitir una conclusión basada en resultados medidos.

> Este laboratorio no exige modificar la arquitectura del agente ni crear agentes adicionales. El objetivo es evaluar y mejorar una implementación existente.

## 1. Preparar Google Colab

En el panel **Secrets** de Google Colab, habilita el acceso a estas cuatro claves: `LLM_API_KEY`, `LLM_MODEL_SMALL`, `LLM_BASE_URL`, `LANGSMITH_API_KEY`.

La configuración se **lee exclusivamente desde Colab Secrets**: no se consultan archivos `.env` ni variables de entorno como origen de credenciales. La biblioteca LangSmith sí utiliza variables de entorno que configuraremos explícitamente a partir de esos secretos.

No imprimas ni copies los valores de las claves en resultados, capturas o repositorios.

In [ ]:
# Instalamos las librerías utilizadas en este laboratorio.
# pandas: agrupar mediciones y crear tablas de comparación.
# matplotlib: visualizar las métricas obtenidas.
# langchain-openai: conexión de ChatOpenAI con la API compatible de Mistral.
# langsmith: registrar las operaciones en una plataforma de trazabilidad.
%pip -q install -U langchain-openai langsmith pandas matplotlib

In [ ]:
import os
import re
import time
import json
import uuid
from datetime import datetime, timezone

import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langsmith import traceable


def obtener_secreto(nombre):
    """Lee un secreto de Colab y falla con un mensaje sin revelar su contenido."""
    try:
        valor = userdata.get(nombre)
    except Exception as exc:
        raise RuntimeError(f"No se pudo leer el secreto {nombre} de Colab.") from exc
    if not isinstance(valor, str) or not valor.strip():
        raise RuntimeError(f"El secreto {nombre} no está configurado o está vacío.")
    return valor.strip()

# Todas las fuentes de configuración son SECRETOS de Colab.
LLM_API_KEY = obtener_secreto("LLM_API_KEY")
LLM_MODEL_SMALL = obtener_secreto("LLM_MODEL_SMALL")
LLM_BASE_URL = obtener_secreto("LLM_BASE_URL")
LANGSMITH_API_KEY = obtener_secreto("LANGSMITH_API_KEY")

# Estas variables activan la instrumentación automática de LangSmith.
# No se leen desde el entorno: se asignan aquí con datos ya obtenidos desde Secrets.
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "ISY0101-EA3-Observabilidad"

# Mistral ofrece una API compatible con el protocolo de ChatOpenAI.
llm = ChatOpenAI(
    model=LLM_MODEL_SMALL,
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
    temperature=0,
    max_retries=1,
)

print("Configuración cargada correctamente desde Colab Secrets.")
print("Trazabilidad habilitada para el proyecto ISY0101-EA3-Observabilidad.")
# No mostramos el modelo ni el endpoint para mantener el criterio de secretos.

## 2. Problema y diseño de la comparación

MercadoAndes presenta una reducción de pedidos y señales de congestión en sus despachos. La solución consulta dos fuentes locales: **comercial** y **operacional**.

Compararemos dos versiones:

- **Versión A (inicial):** ejecuta una consulta operacional **redundante** y repite una espera artificial asociada a esa consulta.
- **Versión B (optimizada):** consulta los datos operacionales una sola vez y reutiliza el resultado.

Ambas versiones envían al LLM exactamente los **mismos datos de negocio**. Así aislamos un cambio específico en el flujo y evitamos confundir una variación de contenido con una optimización de operaciones.

**Hipótesis de trabajo:** al eliminar la llamada redundante se reducirá la latencia y el número de invocaciones de herramientas, sin perder evidencia de negocio. Las llamadas externas al LLM seguirán teniendo variabilidad natural.

In [ ]:
# Datos FIJOS y ficticios de MercadoAndes, compartidos por ambas versiones.
DATOS_COMERCIALES = {
    "pedidos_trimestre_anterior": 1200,
    "pedidos_trimestre_actual": 1020,
    "abandono_carritos_pct": 32,
    "reclamos_totales": 230,
    "reclamos_entrega_tardia": 138,
}
DATOS_OPERACIONES = {
    "despachos_requeridos_semana": 400,
    "capacidad_despachos_semana": 320,
    "entregas_tardias_pct": 27,
}

# Una espera determinística para simular una herramienta que consulta un servicio lento.
# No representa la latencia del LLM ni una medición de producción.
DEMORA_OPERACIONES_SEG = 0.6

# Preguntas de prueba: repetiremos las mismas en A y B.
CASOS = [
    {"id": "C1", "pregunta": "Resume la evolución de pedidos y el abandono de carritos."},
    {"id": "C2", "pregunta": "Describe el problema de capacidad semanal de despachos."},
    {"id": "C3", "pregunta": "Relaciona reclamos por entregas tardías y capacidad logística, sin afirmar causalidad."},
]

print("Casos de prueba:", len(CASOS))
print("Demora ficticia por consulta operacional:", DEMORA_OPERACIONES_SEG, "segundos")

## 3. Instrumentación de herramientas

Usaremos `@traceable` para registrar individualmente las funciones que consultan datos. Los estudiantes podrán ver cada invocación como una operación hija en LangSmith.

Además contabilizaremos las invocaciones realizadas en cada solicitud. **Contar llamadas** permite detectar redundancias incluso si una herramienta responde tan rápido que su latencia resulta difícil de apreciar.

In [ ]:
@traceable(run_type="tool", name="mercadoandes_consulta_comercial")
def consulta_comercial():
    """Devuelve una copia de los datos comerciales controlados."""
    return dict(DATOS_COMERCIALES)


@traceable(run_type="tool", name="mercadoandes_consulta_operaciones")
def consulta_operaciones():
    """Simula la latencia de consultar un servicio de operaciones."""
    time.sleep(DEMORA_OPERACIONES_SEG)
    return dict(DATOS_OPERACIONES)


def extraer_tokens(respuesta):
    """Recupera tokens reportados por el proveedor; no inventa valores faltantes."""
    # El cliente puede colocar el uso en usage_metadata o response_metadata.
    usage = getattr(respuesta, "usage_metadata", None) or {}
    metadata = getattr(respuesta, "response_metadata", None) or {}
    token_usage = metadata.get("token_usage", {}) or {}

    entrada = usage.get("input_tokens", token_usage.get("prompt_tokens"))
    salida = usage.get("output_tokens", token_usage.get("completion_tokens"))
    total = usage.get("total_tokens", token_usage.get("total_tokens"))
    return {"tokens_entrada": entrada, "tokens_salida": salida, "tokens_total": total}


def crear_prompt(pregunta, datos_comerciales, datos_operaciones):
    """Construye idéntico prompt de negocio para las versiones A y B."""
    evidencia = {
        "comercial": datos_comerciales,
        "operaciones": datos_operaciones,
    }
    return (
        "Eres un analista de MercadoAndes. Responde brevemente en español y con lenguaje profesional. "
        "Utiliza SOLO las cifras siguientes, sin inventar datos ni afirmar causalidad. "
        "Distingue observaciones de hipótesis. Incluye las cifras relevantes para la pregunta.\n\n"
        f"EVIDENCIA: {json.dumps(evidencia, ensure_ascii=False, sort_keys=True)}\n"
        f"PREGUNTA: {pregunta}"
    )

## 4. Ejecutar las versiones A y B

Cada prueba realiza dos consultas de datos y una llamada a Mistral. La versión inicial agrega una **segunda consulta operacional innecesaria**, cuyo resultado se descarta. La versión optimizada reutiliza el primer resultado.

En LangSmith, abre una traza de cada versión: deberías ver **dos spans operacionales en A y uno en B**. Esta diferencia es el cambio de ingeniería que evaluaremos.

El código no sustituye respuestas reales por simulaciones: si falla la API, la prueba queda registrada como fallida y no se inventan tokens ni tiempos del modelo.

In [ ]:
# Almacenaremos una fila de métricas por prueba.
MEDICIONES = []


@traceable(run_type="chain", name="MercadoAndes_Lab3_comparacion")
def ejecutar_version(version, caso):
    """Ejecuta un caso y devuelve métricas observadas sin exponer credenciales."""
    if version not in ("A", "B"):
        raise ValueError("La versión debe ser A o B.")

    # Un identificador permite correlacionar esta medición con una ejecución.
    run_id = str(uuid.uuid4())[:8]
    inicio_total = time.perf_counter()
    llamadas_herramienta = 0
    resultado = {
        "run_id": run_id, "version": version, "caso": caso["id"],
        "pregunta": caso["pregunta"], "estado": "error",
        "latencia_total_s": None, "latencia_llm_s": None,
        "llamadas_herramientas": None,
        "tokens_entrada": None, "tokens_salida": None, "tokens_total": None,
        "respuesta": None, "error_tipo": None,
        "datos_integrados_correctos": False,
    }

    try:
        # 1) Consulta comercial: necesaria en ambas versiones.
        comercial = consulta_comercial()
        llamadas_herramienta += 1

        # 2) Consulta operacional: necesaria en ambas versiones.
        operaciones = consulta_operaciones()
        llamadas_herramienta += 1

        # 3) Problema deliberado: A vuelve a pedir lo que ya tiene.
        # El segundo resultado no se utiliza, por lo que es trabajo redundante.
        if version == "A":
            _ = consulta_operaciones()
            llamadas_herramienta += 1

        # 4) El contenido enviado al LLM es equivalente en A y B.
        prompt = crear_prompt(caso["pregunta"], comercial, operaciones)
        resultado["datos_integrados_correctos"] = (
            comercial == DATOS_COMERCIALES and operaciones == DATOS_OPERACIONES
        )

        # 5) Medimos la llamada del modelo por separado de la duración total.
        inicio_llm = time.perf_counter()
        try:
            respuesta_llm = llm.invoke(prompt)
        finally:
            resultado["latencia_llm_s"] = round(time.perf_counter() - inicio_llm, 3)

        resultado["respuesta"] = str(respuesta_llm.content)
        resultado.update(extraer_tokens(respuesta_llm))
        resultado["estado"] = "ok"

    except Exception as exc:
        # Registramos SOLO la clase del error; el texto HTTP podría incluir secretos.
        resultado["error_tipo"] = type(exc).__name__
    finally:
        resultado["latencia_total_s"] = round(time.perf_counter() - inicio_total, 3)
        resultado["llamadas_herramientas"] = llamadas_herramienta

    return resultado

## 5. Medición de los casos de prueba

Ejecutaremos **tres preguntas por versión**, sin cambiar datos, modelo ni instrucciones del LLM.

- El orden de las pruebas puede influir en la latencia externa. Por eso utilizamos las mediciones observadas y evitamos presentar una única ejecución como prueba universal.
- El menor tiempo total es relevante **solo si no empeoran la calidad y los resultados**.
- Si aparece un error de cuota o conectividad, registra el fallo y analiza las pruebas exitosas; no lo reemplaces por una respuesta ficticia.

In [ ]:
# Ejecutamos el mismo conjunto en ambas implementaciones.
# Cada caso se ejecuta en A y B, reduciendo diferencias por el tipo de pregunta.
for caso in CASOS:
    for version in ("A", "B"):
        fila = ejecutar_version(version, caso)
        MEDICIONES.append(fila)
        print(
            f"{fila['caso']} | {version} | {fila['estado']} | "
            f"{fila['latencia_total_s']} s | "
            f"{fila['llamadas_herramientas']} herramientas"
        )

# No imprimimos prompts, credenciales ni respuestas completas por defecto.
df = pd.DataFrame(MEDICIONES)
display(df[["caso", "version", "estado", "latencia_total_s",
            "latencia_llm_s", "llamadas_herramientas", "tokens_total", "error_tipo"]])

## 6. Construcción e interpretación de KPIs

Calcularemos indicadores que pueden defenderse con datos registrados:

- **Latencia total promedio:** tiempo entre el inicio y fin de la solicitud.
- **Latencia LLM:** tiempo consumido específicamente por la llamada al modelo.
- **Llamadas a herramientas:** cantidad de invocaciones por solicitud.
- **Tasa de éxito técnico:** proporción de solicitudes con ejecución completa sin excepción.
- **Tokens reportados:** consumo real informado por el proveedor cuando está disponible.

**Advertencia:** no inferiremos precisión o relevancia únicamente a partir del estado `ok`. Además verificaremos que ambas versiones utilizaron los datos correctos y revisaremos las respuestas frente a las preguntas.

In [ ]:
# Para latencia y consumo del modelo, analizamos únicamente solicitudes exitosas.
# Para tasa de éxito, contamos TODAS las solicitudes, incluidas las fallidas.
exitos = df[df["estado"] == "ok"].copy()

resumen = df.groupby("version").agg(
    solicitudes=("caso", "count"),
    completadas=("estado", lambda serie: int((serie == "ok").sum())),
    llamadas_promedio=("llamadas_herramientas", "mean"),
    integridad_datos=("datos_integrados_correctos", "mean"),
).reset_index()

resumen["exito_tecnico_pct"] = (
    100 * resumen["completadas"] / resumen["solicitudes"]
).round(1)

if not exitos.empty:
    metricas_exito = exitos.groupby("version").agg(
        latencia_promedio_s=("latencia_total_s", "mean"),
        latencia_llm_promedio_s=("latencia_llm_s", "mean"),
        tokens_promedio=("tokens_total", "mean"),
    ).reset_index()
    resumen = resumen.merge(metricas_exito, on="version", how="left")
else:
    resumen["latencia_promedio_s"] = float("nan")
    resumen["latencia_llm_promedio_s"] = float("nan")
    resumen["tokens_promedio"] = float("nan")

# Nota: NaN significa que faltan datos, no que el consumo haya sido cero.
display(resumen.round(3))

In [ ]:
# Visualización 1: latencia TOTAL por caso y versión, solo ejecuciones exitosas.
if not exitos.empty:
    tabla = exitos.pivot_table(index="caso", columns="version", values="latencia_total_s")
    ax = tabla.plot(kind="bar", figsize=(8, 4), rot=0)
    ax.set_title("Latencia medida por caso: versión A vs. B")
    ax.set_xlabel("Caso de prueba")
    ax.set_ylabel("Segundos")
    plt.tight_layout()
    plt.show()
else:
    print("No hay ejecuciones exitosas: no se puede graficar latencia de respuesta.")

# Visualización 2: número de llamadas a herramientas, incluso si falló el LLM.
fig, ax = plt.subplots(figsize=(7, 3.5))
for version, subset in df.groupby("version"):
    ax.plot(subset["caso"], subset["llamadas_herramientas"], marker="o", label=f"Versión {version}")
ax.set_xlabel("Caso de prueba")
ax.set_ylabel("Cantidad de invocaciones")
ax.set_title("Uso de herramientas por versión")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Verificación de calidad de los resultados

Una modificación es útil si reduce trabajo innecesario **sin eliminar evidencia necesaria ni degradar la respuesta**.

El notebook comprueba automáticamente que ambas versiones integran los mismos registros de negocio. Eso no valida por sí solo la calidad del texto generado por Mistral: la exactitud, relevancia y prudencia de las conclusiones deben revisarse con una pauta explícita.

Para cada caso exitoso, revisa si la respuesta: (a) utiliza cifras pertinentes, (b) responde la pregunta y (c) evita presentar asociaciones como causas demostradas.

In [ ]:
# Pauta de revisión MANUAL: cada estudiante puede contrastar respuestas A/B.
# No se asigna calidad = 100 % por el mero hecho de no haber excepciones.
revision = df[["caso", "version", "estado", "datos_integrados_correctos", "respuesta"]].copy()
revision["cifras_pertinentes"] = pd.NA  # Completar con Sí / No tras leer la respuesta.
revision["responde_la_pregunta"] = pd.NA
revision["evita_causalidad_injustificada"] = pd.NA

# Mostramos una respuesta a la vez para facilitar la revisión, sin modificar el dato.
for _, fila in revision.iterrows():
    print(f"\n--- Caso {fila['caso']} | versión {fila['version']} | {fila['estado']} ---")
    print((fila["respuesta"] or "Sin respuesta: ejecución fallida.")[:1200])

display(revision.drop(columns=["respuesta"]))

## 8. Evidencia en LangSmith

Accede a [LangSmith](https://smith.langchain.com/) y abre el proyecto `ISY0101-EA3-Observabilidad`.

1. Busca las operaciones `MercadoAndes_Lab3_comparacion`.
2. Abre una ejecución de la versión A y observa sus llamadas a `mercadoandes_consulta_operaciones`.
3. Contrástala con una ejecución B del mismo caso.
4. Revisa los tiempos y la llamada al modelo. ¿Cuál es la parte que cambió realmente?
5. Si una prueba falló, identifica la operación y verifica qué métricas quedaron sin información.

Los tiempos de la API no son completamente reproducibles: el propósito es **identificar operaciones concretas** y respaldar las conclusiones con medidas, no obtener siempre los mismos números.

## 9. Guardar mediciones

Descarga el CSV producido por esta celda para conservar evidencia del experimento. **No contiene credenciales ni prompts completos**, pero sí etiquetas de casos y métricas de ejecución.

In [ ]:
# Exportamos únicamente columnas de métricas (no respuestas libres ni claves API).
columnas_exportacion = [
    "run_id", "version", "caso", "estado", "latencia_total_s", "latencia_llm_s",
    "llamadas_herramientas", "tokens_entrada", "tokens_salida", "tokens_total",
    "datos_integrados_correctos", "error_tipo",
]
ruta_csv = "mercadoandes_lab3_comparacion.csv"
df[columnas_exportacion].to_csv(ruta_csv, index=False)
print("Métricas guardadas en:", ruta_csv)

## 10. Preguntas de análisis

Responde a partir de tu tabla de KPIs y las trazas de LangSmith:

1. ¿Qué diferencia observaste en la cantidad de llamadas a herramientas entre las versiones A y B? ¿Qué operación explica esa diferencia?
2. ¿Cuánto cambió la latencia total por caso? ¿El cambio fue similar en todas las consultas?
3. ¿Por qué la latencia del LLM puede variar incluso cuando no modificamos el modelo ni el prompt?
4. ¿Cuál es la diferencia entre **tasa de éxito técnico**, **integridad de datos** y **calidad de respuesta**?
5. ¿Qué puedes concluir del consumo de tokens? ¿Qué debes indicar si ese dato no fue reportado?
6. ¿Qué evidencia concreta en LangSmith utilizarías para justificar que la optimización eliminó trabajo redundante?
7. ¿Qué limitaciones tiene esta comparación antes de afirmar que la nueva versión es mejor para producción?

**Conclusión:** redacta un párrafo con el formato **Hallazgo → Evidencia → Modificación → Resultado → Limitación**. Evita conclusiones basadas únicamente en impresiones.

## 11. Cierre

En este laboratorio utilizamos la observabilidad para **respaldar decisiones de ingeniería**. Aprendimos que una mejora debe verificarse en más de una dimensión: latencia, eficiencia operativa, errores, consumo y calidad.

Los datos de negocio de MercadoAndes son ficticios y la espera de la herramienta es intencional. Las llamadas al modelo y sus duraciones, en cambio, se miden durante la ejecución real.

**Referencias para profundizar:** [LangSmith – Observability](https://docs.langchain.com/langsmith/observability) · [LangChain – Tracing](https://docs.langchain.com/langsmith/trace-with-langchain).